# Full Pipeline: Raw Data -> Tuned Multi-Model Ensemble

## Goal

Self-contained, start to finish: raw NASA files -> validation -> RUL generation ->
feature engineering -> feature selection -> train/validation split -> scaling ->
hyperparameter tuning (CatBoost + XGBoost + LightGBM, all three, not just CatBoost) ->
weighted ensemble -> evaluation on validation **and** the official test set.

**Does not assume any prior artifact exists** — `train_prepared.csv`,
`selected_features.json`, `feature_scaler.pkl` are all rebuilt from the raw files in this
notebook, not loaded from disk. Safe to run after a clean checkout or a deleted
`artifacts/` folder — verified by testing it with `artifacts/` deleted before running.

Every tuning trial is automatically logged to MLflow (`BaseTrainer`'s built-in tracking)
— no separate logging step.

**Adjust `N_TRIALS` below before a real run** — defaults low here for a quick correctness
check; bump to 20-30+ per model for a real search.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
print(PROJECT_ROOT)

A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL


In [2]:
N_TRIALS = 20  # per model -- increase for a real search
RUN_TEST_EVAL = True  # official test-set evaluation at the end

In [3]:
import json
import joblib
import pandas as pd

from src.config.config import (
    TRAIN_DATA_PATH, TEST_DATA_PATH, RUL_DATA_PATH,
    MODELS_DIR, REPORTS_DIR, SCALERS_DIR, SELECTED_FEATURES_PATH,
    VALIDATION_SIZE, RANDOM_STATE, DEFAULT_RUL_CAP,
    ROLLING_WINDOW, LAGS, ENGINE_COLUMN, TARGET_COLUMN,
)
from src.utils.constant import SENSOR_COLUMNS
from src.data.loader import DataLoader
from src.data.validator import DataValidator
from src.preprocessing.rul_generator import RULGenerator
from src.preprocessing.feature_engineer import FeatureEngineer
from src.preprocessing.data_splitter import DataSplitter
from src.preprocessing.feature_scaler import FeatureScaler
from src.explainability.feature_selector import FeatureCategorySelector
from src.explainability.feature_reducer import FeatureReducer
from src.optimization.hyperparameter_tuner import ModelTuner
from src.models.model_factory import ModelFactory
from src.models.base_trainer import BaseTrainer
from src.models.ensemble import EnsembleModel
from src.evaluation.evaluator import RegressionEvaluator

a:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1- Load Raw Data

In [4]:
loader = DataLoader(train_path=TRAIN_DATA_PATH, test_path=TEST_DATA_PATH, rul_path=RUL_DATA_PATH)
train_raw = loader.load_train()
test_raw = loader.load_test()
rul_raw = loader.load_rul()

print(f"train_FD004: {train_raw.shape}  test_FD004: {test_raw.shape}  RUL_FD004: {rul_raw.shape}")

2026-09-01 11:59:12 | INFO | loader.py | Line:18 | Reading train_FD004.txt
2026-09-01 11:59:15 | INFO | loader.py | Line:21 | train_FD004.txt Loaded Successfully
2026-09-01 11:59:15 | INFO | loader.py | Line:18 | Reading test_FD004.txt
2026-09-01 11:59:17 | INFO | loader.py | Line:21 | test_FD004.txt Loaded Successfully
2026-09-01 11:59:17 | INFO | loader.py | Line:18 | Reading RUL_FD004.txt
2026-09-01 11:59:17 | INFO | loader.py | Line:21 | RUL_FD004.txt Loaded Successfully


train_FD004: (61249, 26)  test_FD004: (41214, 26)  RUL_FD004: (248, 1)


## 2- Validate

In [5]:
report = DataValidator(train_raw, test_raw, rul_raw).validate_all()
report

2026-09-01 11:59:22 | INFO | validator.py | Line:40 | Validating training dataset...
2026-09-01 11:59:22 | INFO | validator.py | Line:50 | Validating testing dataset...
2026-09-01 11:59:22 | INFO | validator.py | Line:60 | Validating RUL dataset...


{'train': {'valid': True, 'errors': [], 'warnings': []},
 'test': {'valid': True, 'errors': [], 'warnings': []},
 'rul': {'valid': True, 'errors': [], 'warnings': ['Duplicate rows found.']}}

## 3- Generate RUL

In [6]:
train_with_rul = RULGenerator(train_raw).generate(cap=DEFAULT_RUL_CAP)
print(f"cap={DEFAULT_RUL_CAP}  RUL range: [{train_with_rul[TARGET_COLUMN].min()}, {train_with_rul[TARGET_COLUMN].max()}]")

2026-09-01 12:04:41 | INFO | rul_generator.py | Line:82 | Generating Remaining Useful Life (RUL)...
2026-09-01 12:04:41 | INFO | rul_generator.py | Line:92 | Applying RUL cap = 150
2026-09-01 12:04:41 | INFO | rul_generator.py | Line:96 | RUL generated successfully.


cap=150  RUL range: [0, 150]


## 4- Feature Engineering

In [7]:
engineer = FeatureEngineer(sensor_columns=SENSOR_COLUMNS, rolling_window=ROLLING_WINDOW, lags=LAGS)
feature_df = engineer.transform(train_with_rul)
print(f"Engineered: {feature_df.shape[1]} columns ({feature_df.shape[1] - 2} features)")

2026-09-01 12:05:11 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-09-01 12:05:11 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-09-01 12:05:13 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-09-01 12:05:15 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

Engineered: 153 columns (151 features)


## 5- Feature Selection

Reproduces Sprint 10's real, already-validated result (removing rolling features was the
best of 9 real experiments) rather than re-running the full experiment sweep — same
decision, deterministically reapplied.

In [8]:
all_columns = [c for c in feature_df.columns if c not in (ENGINE_COLUMN, TARGET_COLUMN)]
final_features = FeatureCategorySelector.exclude(all_columns, categories=["rolling"])
print(f"Selected {len(final_features)} features")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
reducer = FeatureReducer(keep_features=final_features)
reducer.fit(feature_df[all_columns])
reducer.save_selected_features(SELECTED_FEATURES_PATH)

2026-09-01 12:05:35 | INFO | feature_selector.py | Line:114 | FeatureCategorySelector.exclude(['rolling']): dropped 42, kept 109 features.
2026-09-01 12:05:35 | INFO | feature_reducer.py | Line:101 | FeatureReducer: kept 109 features, removed 42.
2026-09-01 12:05:35 | INFO | feature_reducer.py | Line:154 | Selected feature list saved to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\selected_features.json


Selected 109 features


## 6- Train / Validation Split — by Engine, Never by Row

In [9]:
splitter = DataSplitter(test_size=VALIDATION_SIZE, engine_column=ENGINE_COLUMN, random_state=RANDOM_STATE)
train_split, val_split = splitter.split(feature_df)

print(f"Train: {train_split[ENGINE_COLUMN].nunique()} engines ({train_split.shape[0]} rows)")
print(f"Val  : {val_split[ENGINE_COLUMN].nunique()} engines ({val_split.shape[0]} rows)")

2026-09-01 12:06:11 | INFO | data_splitter.py | Line:35 | Starting engine-based train/validation split...
2026-09-01 12:06:13 | INFO | data_splitter.py | Line:67 | Train Engines: 199 | Validation Engines: 50
2026-09-01 12:06:13 | INFO | data_splitter.py | Line:72 | Data splitting completed successfully.


Train: 199 engines (49294 rows)
Val  : 50 engines (11955 rows)


## 7- Scaling — Fit on Train Only

In [10]:
scaler = FeatureScaler()
X_train = scaler.fit_transform(train_split[final_features])
X_val = scaler.transform(val_split[final_features])
y_train = train_split[TARGET_COLUMN].reset_index(drop=True)
y_val = val_split[TARGET_COLUMN].reset_index(drop=True)

SCALERS_DIR.mkdir(parents=True, exist_ok=True)
scaler.save(SCALERS_DIR / "feature_scaler.pkl")
print(f"X_train: {X_train.shape}  X_val: {X_val.shape}")

2026-09-01 12:06:27 | INFO | feature_scaler.py | Line:33 | Fitting Feature Scaler...


2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:37 | Feature Scaler fitted successfully.
2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:44 | Transforming features...
2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:60 | Feature transformation completed.
2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:44 | Transforming features...
2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:60 | Feature transformation completed.
2026-09-01 12:06:28 | INFO | feature_scaler.py | Line:89 | Scaler saved to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\scalers\feature_scaler.pkl


X_train: (49294, 109)  X_val: (11955, 109)


## 8- Tune Each Model

Same `ModelTuner`, same mechanism, model-specific search space (depth, learning rate,
iterations/trees, regularization, subsampling). Every trial auto-logs to MLflow.

In [11]:
evaluator = RegressionEvaluator()
tuned_models = {}
val_results = []

for model_name in ["catboost", "xgboost", "lightgbm"]:

    print(f"--- Tuning {model_name} ---")
    tuner = ModelTuner(model_name, X_train, y_train, X_val, y_val, random_state=RANDOM_STATE)
    tuner.run(n_trials=N_TRIALS, show_progress_bar=True)

    best_params = tuner.best_params()
    print(f"{model_name} best params: {best_params}")

    final_model = ModelFactory.create(model_name, **best_params)
    final_trainer = BaseTrainer(
        final_model,
        run_name=f"{model_name}_final_tuned",
        tags={"model_family": model_name, "stage": "final_tuned_model"},
    )
    metrics = final_trainer.train(X_train, y_train, X_val, y_val)
    print(f"{model_name} final validation metrics: {metrics}\n")

    tuned_models[model_name] = final_model
    val_results.append({"Model": model_name, "Stage": "tuned_individual", **metrics})

    joblib.dump(final_model, MODELS_DIR / f"{model_name}_tuned.pkl")
    with open(MODELS_DIR / f"{model_name}_best_params.json", "w") as f:
        json.dump({"params": best_params, "metrics": metrics}, f, indent=2)

[I 2026-09-01 12:06:33,614] A new study created in memory with name: catboost_rul_optimization
2026-09-01 12:06:33 | INFO | hyperparameter_tuner.py | Line:142 | Starting Optuna search for catboost: 20 trials, search space = ['depth', 'learning_rate', 'iterations', 'l2_leaf_reg', 'subsample', 'random_strength']


--- Tuning catboost ---


  0%|          | 0/20 [00:00<?, ?it/s]2026/09/01 12:07:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/09/01 12:07:23 INFO mlflow.store.db.utils: Updating database tables
2026/09/01 12:07:29 INFO mlflow.tracking.fluent: Experiment with name 'predictive-maintenance-rul' does not exist. Creating a new experiment.
2026-09-01 12:07:29 | INFO | base_trainer.py | Line:74 | Training CatBoostRegressor...
2026-09-01 12:07:52 | INFO | base_trainer.py | Line:80 | Training completed successfully in 23.23s.
2026-09-01 12:07:52 | INFO | base_trainer.py | Line:129 | Generating predictions using CatBoostRegressor...
2026-09-01 12:07:52 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-09-01 12:07:52 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026/09/01 12:07:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 12:08:14 WARNING mlflow.utils.environment: Failed to resolve insta

[I 2026-09-01 12:08:14,456] Trial 0 finished with value: 18.18240061904868 and parameters: {'depth': 6, 'learning_rate': 0.2536999076681772, 'iterations': 1152, 'l2_leaf_reg': 6.387926357773329, 'subsample': 0.5780093202212182, 'random_strength': 1.5599452033620265}. Best is trial 0 with value: 18.18240061904868.


2026-09-01 12:08:25 | INFO | base_trainer.py | Line:80 | Training completed successfully in 10.84s.
2026-09-01 12:08:25 | INFO | base_trainer.py | Line:129 | Generating predictions using CatBoostRegressor...
2026-09-01 12:08:25 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-09-01 12:08:25 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026/09/01 12:08:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 12:08:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026-09-01 12:08:35 | INFO | base_trainer.py | Line:99 | Logged to MLflow: run_id=a577188716034963864e7a68a9ba1565
2026-09-01 12:08:35 | INFO | hyperparameter_tuner.py | Line:219 | [catboost] Trial 1 | MAE=18.1402 | RMSE=25.0335 | R2=0.7505 | Time=21.45s | params={'depth': 4, 'learning_rate': 0.19030368381735815, 'iterations': 98

...

[I 2026-09-01 13:45:56,240] Trial 18 finished with value: 17.60819845289184 and parameters: {'max_depth': 8, 'num_leaves': 120, 'learning_rate': 0.017850705608912134, 'n_estimators': 539, 'reg_lambda': 7.497509325046325, 'subsample': 0.8535972392831754, 'colsample_bytree': 0.6929057826848473}. Best is trial 14 with value: 17.462308167725237.


2026-09-01 13:46:11 | INFO | base_trainer.py | Line:80 | Training completed successfully in 14.87s.
2026-09-01 13:46:11 | INFO | base_trainer.py | Line:129 | Generating predictions using LGBMRegressor...
2026-09-01 13:46:11 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-09-01 13:46:11 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026/09/01 13:46:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 13:46:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026-09-01 13:46:22 | INFO | base_trainer.py | Line:99 | Logged to MLflow: run_id=70fc3471051145d397f0c34a3f1bc153
2026-09-01 13:46:22 | INFO | hyperparameter_tuner.py | Line:219 | [lightgbm] Trial 19 | MAE=17.4857 | RMSE=24.6843 | R2=0.7574 | Time=26.27s | params={'max_depth': 8, 'num_leaves': 212, 'learning_rate': 0.042140975298617

[I 2026-09-01 13:46:22,548] Trial 19 finished with value: 17.485681517348272 and parameters: {'max_depth': 8, 'num_leaves': 212, 'learning_rate': 0.042140975298617024, 'n_estimators': 852, 'reg_lambda': 3.1352748603351666, 'subsample': 0.8404081163966654, 'colsample_bytree': 0.582027767988615}. Best is trial 14 with value: 17.462308167725237.
lightgbm best params: {'random_state': 42, 'verbose': -1, 'max_depth': 10, 'num_leaves': 188, 'learning_rate': 0.015401818885932124, 'n_estimators': 687, 'reg_lambda': 3.0250870556045992, 'subsample': 0.6980786230998579, 'colsample_bytree': 0.5562286377634573}


2026-09-01 13:46:41 | INFO | base_trainer.py | Line:80 | Training completed successfully in 19.15s.
2026-09-01 13:46:41 | INFO | base_trainer.py | Line:129 | Generating predictions using LGBMRegressor...
2026-09-01 13:46:42 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-09-01 13:46:42 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.
2026/09/01 13:46:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/01 13:46:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026-09-01 13:46:53 | INFO | base_trainer.py | Line:99 | Logged to MLflow: run_id=133fdc0242984037937926d646add768


lightgbm final validation metrics: {'MAE': 17.462308167725237, 'RMSE': 24.576807679942657, 'R2': 0.7594715574010924, 'MAPE': 27.375110158213324, 'Training Time (s)': 19.15}



## 9- Check the Tuning Runs in MLflow

Every trial from every model, queryable in one place.

In [12]:
import mlflow
from src.experiments.mlflow_tracker import MLflowTracker

tracker = MLflowTracker()
runs = mlflow.search_runs(
    filter_string="tags.stage = 'hyperparameter_search'",
    order_by=["metrics.MAE ASC"],
)
runs[["tags.model_family", "tags.mlflow.runName", "metrics.MAE", "metrics.RMSE", "metrics.R2"]].head(10)

,tags.model_family,tags.mlflow.runName,metrics.MAE,metrics.RMSE,metrics.R2
0,xgboost,xgboost_optuna_trial_14,17.347773,24.713768,0.756783
1,xgboost,xgboost_optuna_trial_12,17.352707,24.733965,0.756386
2,xgboost,xgboost_optuna_trial_15,17.374544,24.716388,0.756732
3,xgboost,xgboost_optuna_trial_18,17.382746,24.767436,0.755726
4,xgboost,xgboost_optuna_trial_13,17.383631,24.687090,0.757308
5,xgboost,xgboost_optuna_trial_16,17.420092,24.763927,0.755795
6,xgboost,xgboost_optuna_trial_8,17.457476,24.716056,0.756738
7,lightgbm,lightgbm_optuna_trial_14,17.462308,24.576808,0.759472
8,xgboost,xgboost_optuna_trial_2,17.466625,24.713973,0.756779
9,lightgbm,lightgbm_optuna_trial_17,17.467085,24.694491,0.757163


## 10- Build the Ensemble

Weighted by each member's inverse validation MAE — a more accurate member counts for
more, rather than a flat average.

In [13]:
val_mae_by_model = {r["Model"]: r["MAE"] for r in val_results}
ensemble = EnsembleModel.from_inverse_mae(tuned_models, val_mae_by_model)

ensemble_val_preds = ensemble.predict(X_val)
ensemble_val_metrics = evaluator.evaluate(y_val, ensemble_val_preds)
print(f"Ensemble weights: {ensemble.weights}")
print(f"Ensemble validation metrics: {ensemble_val_metrics}")

val_results.append({"Model": "ensemble", "Stage": "tuned_ensemble", **ensemble_val_metrics})

ENSEMBLE_DIR = MODELS_DIR / "ensemble"
ENSEMBLE_DIR.mkdir(parents=True, exist_ok=True)
ensemble.save(ENSEMBLE_DIR)
print(f"Saved -> {ENSEMBLE_DIR}")

2026-09-01 13:47:58 | INFO | ensemble.py | Line:56 | EnsembleModel created: ['catboost', 'xgboost', 'lightgbm'] | weights={'catboost': 0.3295140386994192, 'xgboost': 0.33634603025117793, 'lightgbm': 0.3341399310494028}
2026-09-01 13:47:58 | INFO | evaluator.py | Line:47 | Starting regression evaluation...
2026-09-01 13:47:58 | INFO | evaluator.py | Line:80 | Evaluation completed successfully.


Ensemble weights: {'catboost': 0.3295140386994192, 'xgboost': 0.33634603025117793, 'lightgbm': 0.3341399310494028}
Ensemble validation metrics: {'MAE': 17.370733586947733, 'RMSE': 24.574472980427412, 'R2': 0.7595172537317618, 'MAPE': 26.90317287904684}


2026-09-01 13:47:59 | INFO | ensemble.py | Line:97 | EnsembleModel saved to A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\ensemble


Saved -> A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\artifacts\models\ensemble


## 11- Validation Comparison

In [14]:
val_results_df = pd.DataFrame(val_results)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
val_results_df.to_csv(REPORTS_DIR / "tuned_ensemble_validation_results.csv", index=False)
val_results_df.sort_values("MAE")

,Model,Stage,MAE,RMSE,R2,MAPE,Training Time (s)
1,xgboost,tuned_individual,17.347773,24.713768,0.756783,26.821395,146.82
3,ensemble,tuned_ensemble,17.370734,24.574473,0.759517,26.903173,NaN
2,lightgbm,tuned_individual,17.462308,24.576808,0.759472,27.375110,19.15
0,catboost,tuned_individual,17.707453,24.836150,0.754369,28.146478,235.56


## 12- Official Test-Set Evaluation

Reuses `test_raw`/`rul_raw` loaded in Section 1 — no reloading. Same feature engineering
object, same fitted scaler, same 109 features. No retraining.

In [15]:
if RUN_TEST_EVAL:
    test_features_df = engineer.transform(test_raw)
    last_rows = (
        test_features_df.sort_values([ENGINE_COLUMN, "time_in_cycles"])
        .groupby(ENGINE_COLUMN).tail(1).sort_values(ENGINE_COLUMN).reset_index(drop=True)
    )

    X_test = scaler.transform(last_rows[final_features])
    y_test_true = rul_raw["RUL"].to_numpy()

    test_results = []
    for model_name, model in tuned_models.items():
        preds = model.predict(X_test)
        metrics = evaluator.evaluate(y_test_true, preds)
        test_results.append({"Model": model_name, "Stage": "tuned_individual", **metrics})

    ensemble_test_preds = ensemble.predict(X_test)
    ensemble_test_metrics = evaluator.evaluate(y_test_true, ensemble_test_preds)
    test_results.append({"Model": "ensemble", "Stage": "tuned_ensemble", **ensemble_test_metrics})

    test_results_df = pd.DataFrame(test_results)
    test_results_df.to_csv(REPORTS_DIR / "tuned_ensemble_test_results.csv", index=False)
    display(test_results_df.sort_values("MAE"))
else:
    print("Skipped (RUN_TEST_EVAL=False)")

2026-09-01 13:48:32 | INFO | feature_engineer.py | Line:50 | Starting Feature Engineering...
2026-09-01 13:48:32 | INFO | feature_engineer.py | Line:119 | Generating Rolling Mean features...
2026-09-01 13:48:33 | INFO | feature_engineer.py | Line:148 | Generating Rolling Std features...
2026-09-01 13:48:34 | INFO | feature_engineer.py | Line:179 | Generating Lag Features...
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[feature_name] = (
A:\AI_Engineer\ML Projects\Predictive-Maintenance-RUL\src\preprocessing\feature_engineer.py:189: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has p

,Model,Stage,MAE,RMSE,R2,MAPE
3,ensemble,tuned_ensemble,19.226119,25.630407,0.779027,27.247437
0,catboost,tuned_individual,19.235581,25.711463,0.777627,28.620949
2,lightgbm,tuned_individual,19.381339,25.715904,0.777550,27.482915
1,xgboost,tuned_individual,19.502796,25.908839,0.774200,27.995096


## Conclusion

- **Verified fully self-contained**: tested with `artifacts/` deleted beforehand, ran clean from raw data through to the final comparison, zero errors.
- **`N_TRIALS` is set to 20** for a real run — increase to 30-50 if you have the time budget. Each trial takes roughly 5-40s depending on the model and sampled hyperparameters.
- Compare the validation and test tables: promote the ensemble only if it wins on the **test** table specifically — validation-only comparison isn't sufficient (see Sprint 14's finding on why validation MAE can be misleading across configs).
- Every trial is in MLflow regardless of which model wins, so nothing is lost even if you rerun with different settings later.